In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

sys.append

from config.constants import PROJECT_ROOT
%load_ext autoreload
%autoreload 2

In [ ]:
root_dir = "/Users/zy51nise/Documents/BIONETs/FLabNet/Code-main/FLabBench-pipeline/saved_data/cohorts/DTB/"
sava_dir = "/Users/zy51nise/Documents/BIONETs/FLabNet/Code-main/FLabBench-pipeline/plots/"
sel_edges = pd.read_csv(root_dir + "selected_edges_DTB_all.csv")


#filter cohorts
possible_cohorts = sel_edges[(sel_edges["n_pos"] > 10) & (sel_edges["n_neg"] > 50)]
possible_cohorts["n_cohort"] = possible_cohorts["n_pos"] + possible_cohorts["n_neg"]
possible_cohorts["target_rate"] = possible_cohorts["n_pos"] / possible_cohorts["n_cohort"]
possible_cohorts["log_n_cohort"] = np.log(possible_cohorts["n_cohort"])
possible_cohorts["log_target_rate"] = np.log(possible_cohorts["target_rate"])
#possible_cohorts = possible_cohorts[possible_cohorts["target_rate"] > 0.01]


In [ ]:
print(possible_cohorts[["target_rate", "n_cohort"]].describe())

In [ ]:
plot_df = possible_cohorts.copy()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 10))
axes = axes.ravel()

sns.histplot(plot_df["target_rate"], bins=50, ax=axes[0])
sns.histplot(plot_df["n_cohort"], bins=50, ax=axes[1])
sns.scatterplot(data=plot_df, x="n_cohort", y="target_rate", s=10, alpha=0.4, ax=axes[2])
sns.scatterplot(data=plot_df, x="log_n_cohort", y="log_target_rate", s=10, alpha=0.4, ax=axes[3])


plot_df["D1cat"] = plot_df["D1"].str[:1]
plot_df["D2cat"] = plot_df["D2"].str[:1]
cmap = sns.color_palette("husl", 16)

sns.scatterplot(data=plot_df,x="log_n_cohort", y="log_target_rate", hue="D1cat", palette=cmap, alpha=0.4, legend=True,ax=axes[4])
sns.scatterplot(data=plot_df,x="log_n_cohort", y="log_target_rate", hue="D2cat", palette=cmap, alpha=0.4, legend=True,ax=axes[5])
axes[4].set_title("(D1)")
axes[5].set_title("(D2)")

plt.tight_layout()
plt.savefig(sava_dir + "DTB_cohorts_characteristics.png")
plt.show()

In [ ]:
mapping = {
    "A": "infectious and parasitic",
    "B": "infectious and parasitic",
    "C": "Neoplasms",
    "D": "blood and blood-forming organs",
    "E": "nutritional and metabolic",
    "F": "Mental and behavioural disorders",
    "G": "nervous system",
    "H": "eye and adnexa",
    "I": "circulatory system",
    "J": "respiratory system",
    "K": "digestive system",
    "L": "skin and subcutaneous tissue",
    "M": "musculoskeletal system",
    "N": "genitourinary system",
    "O": "pregnancy",
    "P": "perinatal period",
    "Q": "Congenital malformations",
}


possible_cohorts["D1cat"] = possible_cohorts["D1"].astype(str).str[:1]
possible_cohorts["D2cat"] = possible_cohorts["D2"].astype(str).str[:1]

possible_cohorts["D1type"] = possible_cohorts["D1cat"].map(mapping)
possible_cohorts["D2type"] = possible_cohorts["D2cat"].map(mapping)



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d1 = possible_cohorts["D1type"].value_counts()
d2 = possible_cohorts["D2type"].value_counts()

types = sorted(set(d1.index) | set(d2.index))

# order by max(D1, D2) descending
order = (
    pd.DataFrame({"D1": d1.reindex(types).fillna(0), "D2": d2.reindex(types).fillna(0)})
    .assign(max_=lambda x: x[["D1", "D2"]].max(axis=1))
    .sort_values("max_", ascending=False)
    .index.tolist()
)

D1 = d1.reindex(order).fillna(0).values
D2 = d2.reindex(order).fillna(0).values

y = np.arange(len(order))
h = 0.4  # bar height

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(y - h/2, D1, height=h, color="#A6CEE3", label="D1")
ax.barh(y + h/2, D2, height=h, color="#FDBF6F", label="D2")

ax.set_yticks(y)
ax.set_yticklabels(order)
ax.invert_yaxis()  # largest at top
ax.set_xlabel("count")
ax.set_title(f"Top type counts (D1 vs D2)")
ax.legend()
plt.tight_layout()
#plt.savefig(sava_dir + "DTB_cohorts_D1_D2_counts.png")
plt.show()

In [ ]:
D1_to_D2 = (
    possible_cohorts.groupby("D1type")["D2type"]
    .value_counts()
    .groupby(level=0)
    .head(5)
    .reset_index(name="count")
)
D1_to_D2.sort_values("count", ascending=False)

In [ ]:
k = 5
common = (
    possible_cohorts.groupby("D1type")["D2type"]
    .value_counts()
    .groupby(level=0).head(k)
    .rename("count")
    .reset_index()
)

pivot = common.pivot(index="D1type", columns="D2type", values="count").fillna(0)
colors = sns.color_palette("tab20") 
ax = pivot.plot(kind="bar", stacked=True, figsize=(14,6),color=colors)
ax.set_title(f"Top {k} D2type per D1type")
ax.set_xlabel("D1type")
ax.set_ylabel("count")
plt.legend(title="D2type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## select sample cohorts from each group

In [ ]:
target_metric = "log_target_rate"

In [ ]:
max = possible_cohorts[target_metric].max()
min = possible_cohorts[target_metric].min()
step = (max - min) / 4
bins = [min, min+step, min+2*step, min+3*step, min+4*step]
labels = ["very_low", "low", "medium", "high"]
possible_cohorts["size_group"] = pd.cut(possible_cohorts[target_metric], bins=bins, labels=labels, include_lowest=True)
#possible_cohorts.groupby("size_group")[[target_metric]].describe()

In [ ]:
pd.crosstab(possible_cohorts["size_group"], possible_cohorts["D1type"])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

ct = pd.crosstab(possible_cohorts["size_group"], possible_cohorts["D1type"])

colors = sns.color_palette("tab20", n_colors=ct.shape[1]) 

ax = ct.plot(kind="bar", stacked=True, figsize=(12,5),color=colors)
ax.set_title("D1type counts by size_group")
ax.set_xlabel("size_group")
ax.set_ylabel("count")
plt.legend(title="D1type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
pd.crosstab(possible_cohorts["size_group"], possible_cohorts["D2type"])

In [ ]:
ct = pd.crosstab(possible_cohorts["size_group"], possible_cohorts["D2type"])

colors = sns.color_palette("turbo", n_colors=ct.shape[1]) 

ax = ct.plot(kind="bar", stacked=True, figsize=(12,5),color=colors)
ax.set_title("D2type counts by size_group")
ax.set_xlabel("size_group")
ax.set_ylabel("count")
plt.legend(title="D2type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
high_cohort = possible_cohorts[possible_cohorts["size_group"] == "high"].head(10)
medium_cohort = possible_cohorts[possible_cohorts["size_group"] == "medium"].head(10)
low_cohort = possible_cohorts[possible_cohorts["size_group"] == "low"].head(10)
very_low_cohort = possible_cohorts[possible_cohorts["size_group"] == "very_low"].head(10)


In [ ]:


rows = []
for tier, cohort_df in [
    ("high", high_cohort),
    ("medium", medium_cohort),
    ("low", low_cohort),
    ("very_low", very_low_cohort),
]:
    part = cohort_df[["D1", "D2"]].drop_duplicates().copy()
    part["tier"] = tier
    rows.append(part)

out = pd.concat(rows, ignore_index=True)
out.to_csv(root_dir + f"selected_edges_DTB_{target_metric}_all.csv", index=False)

In [ ]:
target_metric = "log_n_cohort"
out = pd.read_csv(root_dir + f"selected_edges_DTB_{target_metric}_all.csv")


## Results

In [ ]:
import os
import re
from pathlib import Path
dir = Path("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/results/")


In [ ]:
cohort_folders = [p.name for p in Path(dir).iterdir() if p.is_dir() and p.name.startswith("cohort")]
# extract D1, D2 from "cohort_D1_D2_W"
records = []
for name in cohort_folders:
    m = re.match(r"cohort_([^_]+)_([^_]+)_(.+)", name)
    if m:
        records.append({"name": name, "D1": m.group(1), "D2": m.group(2), "W": m.group(3)})
df_folders = pd.DataFrame(records)



In [ ]:
df_folders= df_folders.rename(columns={"name": "cohort"})

In [ ]:
out_edges = possible_cohorts.merge(df_folders[["D1", "D2"]].drop_duplicates(), on=["D1", "D2"], how="inner")
out_edges.rename(columns={"size_group_n_cohort": "cs_group"}, inplace=True)
out_edges.rename(columns={"size_group": "tr_group"}, inplace=True)

In [ ]:
out_edges [["D1", "D2", "RR", "D1type", "D2type", "n_cohort","target_rate","tr_group","cs_group"]]

In [ ]:
df_folders.merge(out_edges[["D1", "D2", "tr_group", "cs_group"]], on=["D1", "D2"], how="inner")

In [ ]:
model ="CatBoost"
results_df = pd.DataFrame()
for cohort in df_folders["cohort"].unique():
    file = dir = Path(f"/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/results/{cohort}/non_time_series/concatenate/minority/feature_selection_True/grid_False/feat_type_VMD/agg_interval_24h/2503/CatBoost_results.csv")
    if file.exists():
        output = pd.read_csv(file)
        output["cohort"] = cohort 
        

        results_df = pd.concat([results_df, output])
results_df =results_df[["cohort","Fold","AUC-ROC", "AUC-PRC", "f1_score"]]
for col in ["AUC-ROC", "AUC-PRC", "f1_score"]:
    results_df[col] = pd.to_numeric(results_df[col], errors="coerce")

for col, new_col in [("AUC-ROC", "roc_avg"), ("AUC-PRC", "pr_avg"), ("f1_score", "f1_avg")]:
    results_df[new_col] = results_df.groupby("cohort")[col].transform("mean")



results_df = results_df[["cohort", "roc_avg", "pr_avg", "f1_avg"]].drop_duplicates("cohort")


In [ ]:
results_df =results_df.merge(df_folders[["cohort","D1", "D2"]], on=["cohort"], how="inner")
results_df =results_df.merge(out_edges[["D1", "D2", "target_rate", "n_cohort", "tr_group", "cs_group","RR"]], on=["D1", "D2"], how="inner")

In [ ]:
results_df.sort_values(["roc_avg"], ascending=False)
results_df.to_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/analysis/results.csv",index=False)

In [ ]:
results_df.groupby("tr_group")["AUC-ROC"].unique()

def parse_mean(val):
    if pd.isna(val):
        return float("nan")
    m = re.match(r"([0-9.]+)\s*[±+-]+", str(val))
    return float(m.group(1)) if m else float("nan")

results_df["AUC-ROC_mean"] = results_df["AUC-ROC"].apply(parse_mean)
results_df["AUC-ROC_std"] = results_df["AUC-ROC"].apply(
    lambda v: float(re.search(r"[±+-]+\s*([0-9.]+)", str(v)).group(1))
    if re.search(r"[±+-]+\s*([0-9.]+)", str(v)) else float("nan")
)
results_df.fillna(0)

results_df.to_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/analysis/results.csv",index=False)

In [ ]:
results_df.groupby(["tr_group"])["AUC-ROC_mean"]

In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

# pick numeric features that describe each cohort
feature_cols = ["target_rate", "n_cohort", "AUC-PRC_mean"]
df_feat = results_df[feature_cols + ["tr_group"]].dropna()

X = StandardScaler().fit_transform(df_feat[feature_cols])
pca = PCA(n_components=2)
coords = pca.fit_transform(X)

df_feat = df_feat.copy()
df_feat["PC1"] = coords[:, 0]
df_feat["PC2"] = coords[:, 1]

order = ["very_low", "low", "medium", "high"]
palette = dict(zip(order, sns.color_palette("Set2", len(order))))

fig, ax = plt.subplots(figsize=(8, 6))
for group in order:
    sub = df_feat[df_feat["tr_group"] == group]
    ax.scatter(sub["PC1"], sub["PC2"], label=group, color=palette[group], alpha=0.6, s=20)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title("PCA of cohort features colored by tr_group")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import umap
reducer = umap.UMAP(n_components=2, random_state=42)
coords = reducer.fit_transform(X)

In [ ]:
import umap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

feature_cols = ["n_cohort", "RR"]
df_feat = out_edges[feature_cols + ["tr_group"] + ["cs_group"]].dropna()

X = StandardScaler().fit_transform(df_feat[feature_cols])
reducer = umap.UMAP(n_components=2, random_state=42)
coords = reducer.fit_transform(X)

df_feat = df_feat.copy()
df_feat["U1"] = coords[:, 0]
df_feat["U2"] = coords[:, 1]

order = ["low", "medium", "high"]
palette = dict(zip(order, sns.color_palette("Set2", len(order))))

fig, ax = plt.subplots(figsize=(4, 3))
for group in order:
    sub = df_feat[df_feat["cs_group"] == group]
    ax.scatter(sub["U1"], sub["U2"], label=group, color=palette[group], alpha=0.6, s=20)

#ax.set_xlabel("UMAP 1")
#ax.set_ylabel("UMAP 2")
plt.title("based on cohort size")
ax.legend()
plt.savefig(sava_dir + "DTB_cohorts_UMAP_cohort_size.png")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


df_merged = results_df.copy()
metric = "AUC-PRC_mean"
group_col = "cs_group"
order = [ "low", "medium", "high"]
palette = dict(zip(order, sns.color_palette("Set2", len(order))))

fig, ax = plt.subplots(figsize=(8, 5))
sns.stripplot(
    data=df_merged.dropna(subset=[metric]),
    x=group_col, y=metric,
    order=order, palette=palette,
    jitter=0.25, alpha=0.5, size=4, ax=ax
)
sns.boxplot(
    data=df_merged.dropna(subset=[metric]),
    x=group_col, y=metric,
    order=order, palette=palette,
    width=0.4, fliersize=0, ax=ax
)
#ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="random baseline")
ax.set_xlabel("Cohort size group", fontsize=11)
ax.set_ylabel(metric, fontsize=11)
ax.set_title("Model performance", fontsize=11)
ax.legend()
plt.tight_layout()
plt.savefig(sava_dir + "DTB_cohorts_AUC-PRC_by_cs_group.png")
plt.show()

In [ ]:
from scipy import stats
import scikit_posthocs as sp

data = df_merged.dropna(subset=["AUC-ROC_mean"])
groups = [grp["AUC-ROC_mean"].values for _, grp in data.groupby("cs_group")]

stat, p = stats.kruskal(*groups)
print(f"Kruskal-Wallis: H={stat:.3f}, p={p:.4f}")

dunn = sp.posthoc_dunn(data, val_col="AUC-ROC_mean", group_col="cs_group", p_adjust="fdr_bh")
print(dunn)

In [ ]:
print("Kruskal-Wallis: H={:.3f}, p={:.4f}".format(stat, p))
print("\nDunn post-hoc p-values (BH corrected):")
print(dunn.round(4))

print("\nSignificantly different pairs (p < 0.05):")
order = ["low", "medium", "high"]
for i in range(len(order)):
    for j in range(i+1, len(order)):
        g1, g2 = order[i], order[j]
        pval = dunn.loc[g1, g2]
        sig = "YES ***" if pval < 0.001 else ("YES *" if pval < 0.05 else "no")
        print(f"  {g1} vs {g2}: p={pval:.4f}  →  {sig}")

# Find largest cohort


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

root_dir = "/Users/zy51nise/Documents/BIONETs/FLabNet/Code/FLabBench-pipeline/data/MIMIC_IV/cohorts/DTB/"
sel_edges = pd.read_csv(root_dir + "selected_edges_DTB_all.csv")

In [ ]:
possible_cohorts = sel_edges[(sel_edges["n_pos"] > 10) & (sel_edges["n_neg"] > 50)]
possible_cohorts["n_cohort"] = possible_cohorts["n_pos"] + possible_cohorts["n_neg"]
possible_cohorts["target_rate"] = possible_cohorts["n_pos"] / possible_cohorts["n_cohort"]
possible_cohorts["log_n_cohort"] = np.log(possible_cohorts["n_cohort"])
possible_cohorts["log_target_rate"] = np.log(possible_cohorts["target_rate"])

In [ ]:
len(possible_cohorts)

In [ ]:
sub_cohorts =possible_cohorts[possible_cohorts['target_rate'] > 0.01]
sub_cohorts.sort_values(by='n_cohort', ascending=False).head(5)

In [ ]:
len(sub_cohorts)

In [ ]:
cohort = pd.read_csv(root_dir + "cohort_E78_H35_791.08.csv.gz", compression="gzip")

In [ ]:
cohort["label"].sum()

In [ ]:
len(cohort)

In [ ]:
cohort